# 14 · Diagnostic: does re-tokenizing generated text shift the self-scoring curvature? (design doc §8.8.23, pre-registered)

**How to run**: pick a Colab server with an **A100 GPU** (an L4 also works, more slowly) and click "Run All". Rough estimate, not measured: 20-30 minutes on an A100. If the connection drops, click "Run All" again; finished models are skipped. Results go to `DATA_DIR/14_retokenization_diagnostic/`.

**What it computes**: notebook 13 stored generated continuations as text, and scoring encoded that text again, so the scored tokens were not always the sampled ones. For the same four models, this notebook generates 300 new continuations at T = 1 (new seed), keeps the sampled token ids, and scores every continuation twice: once exactly as notebook 13 did (text path) and once on the sampled ids (id path). The analysis (`scripts/e11_retokenization_diagnostic.py`) compares the mean curvature mean(H - s) of the two paths.

In [ ]:
# 1. Check for a GPU with bfloat16 support (A100 or L4).
import torch
assert torch.cuda.is_available(), "No GPU: switch the Colab server to an A100 GPU"
name = torch.cuda.get_device_name(0); mem = torch.cuda.get_device_properties(0).total_memory / 2**30
print(name, f"{mem:.0f} GB")
assert torch.cuda.is_bf16_supported(), f"{name} has no bfloat16 support; choose an A100 or L4 GPU"
assert mem > 15, f"This {name} has only {mem:.0f} GB; choose an A100 or L4 GPU"

In [ ]:
# 2. Mount Google Drive and switch to the data folder.
import os
from google.colab import drive
drive.mount("/content/drive")
DATA_DIR = "/content/drive/MyDrive/curvature-margin/data"  # folder with the layout in data/README.md
%cd $DATA_DIR
OUT = "14_retokenization_diagnostic"
os.makedirs(OUT, exist_ok=True)
os.environ["HF_HOME"] = "/content/hf"

In [ ]:
# 3. Install packages (as in 09).
%pip install -q -U transformers huggingface-hub accelerate
import os, json, gc, glob, shutil, time, hashlib, sys, platform
from datetime import datetime, timezone
import numpy as np, pandas as pd
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM

In [ ]:
# 4. Held-out M4GT human texts (as in 09): check the raw file and manifest hashes, then every text hash.
RAW = "external/m4gt/SubtaskB.jsonl"; MAN = "manifests/m4gt_humans_A_manifest.csv"
EXPECTED_RAW = "4d65fc1fb93b5c9d21b226aca06b610ee031fa9ccbc18fa6f39d16273dd281c7"
EXPECTED_MAN = "7c6f2b6529a8ef7b1a1920bc798d45c68a7a7162823566f52a73deb6ae8d1caf"

def sha256_file(path):
    d = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            d.update(block)
    return d.hexdigest()

assert sha256_file(RAW) == EXPECTED_RAW, "Raw file SHA-256 does not match; stopping."
assert sha256_file(MAN) == EXPECTED_MAN, "Human manifest SHA-256 does not match; stopping."
man = pd.read_csv(MAN)
assert len(man) == 1000 and man.id.tolist() == list(range(1000)) and (man.role[:300] == "source").all()
need = {int(r.line_number): r for r in man.itertuples(index=False)}
found = {}
with open(RAW, encoding="utf-8") as f:
    for ln, line in enumerate(f, start=1):
        r = need.get(ln)
        if r is None:
            continue
        it = json.loads(line)
        assert hashlib.sha256(it["text"].encode("utf-8")).hexdigest() == r.text_sha256, f"Text hash mismatch on line {ln}"
        assert it["source"] == r.source and it["model"] == "human"
        found[int(r.id)] = dict(id=int(r.id), domain=r.source, text=it["text"])
assert len(found) == 1000
humans = [found[i] for i in range(1000)]; sources = humans[:300]
print(len(humans), "human texts; prompts:", len(sources))

In [ ]:
# 5. Pre-registered settings (design doc 8.8.23). load, free, bos, prepare and generate_batch are copied from notebook 13;
#    the scoring loop of notebook 13's score_self moved unchanged into score_encoded, so score_self (text path) is identical.
MODELS = {"allenai/OLMo-2-0425-1B": "a1847dff35000b4271fa70afc5db10fd29fedbdf",
          "ibm-granite/granite-3.3-2b-base": "c15dc681f4b2aeb4e576cdaae5434edfc17ed14d",
          "EleutherAI/gpt-neo-1.3B": "dbe59a7f4a88d01d1ba9798d78dbe3fe038792c8",
          "bigscience/bloom-1b7": "cc72a88036c2fb937d65efeacc57a0c2ef5d6fe5"}
TEMP = 1.0
PREFIX, MAX_TOK, BS, SBS, SEED = 30, 512, 32, 16, 20260930   # new seed: fresh samples, not notebook 13's
tag_of = lambda name: name.split("/")[1]

def load(name):
    tok = AutoTokenizer.from_pretrained(name, revision=MODELS[name])
    if tok.pad_token_id is None:
        tok.pad_token = tok.eos_token
    model = AutoModelForCausalLM.from_pretrained(name, revision=MODELS[name], dtype=torch.bfloat16).cuda().eval()
    return tok, model

def free(name):
    gc.collect(); torch.cuda.empty_cache()
    shutil.rmtree(f"/content/hf/hub/models--{name.replace('/', '--')}", ignore_errors=True)

def bos(tok):   # the BOS id if this tokenizer adds one by default (none of the development models did)
    first = tok("a")["input_ids"][:1]
    return first if tok.bos_token_id is not None and first == [tok.bos_token_id] else []

def prepare(tok, texts):   # split on raw text tokens; a default BOS is added again when generating and scoring
    items = []
    for h in texts:
        ids = tok(h["text"], add_special_tokens=False, truncation=True, max_length=MAX_TOK - 1)["input_ids"]
        if len(ids) < PREFIX + 20:
            continue
        items.append(dict(id=h["id"], domain=h["domain"], prefix_ids=ids[:PREFIX], target=len(ids) - PREFIX,
                          human=tok.decode(ids[PREFIX:], clean_up_tokenization_spaces=False),
                          prefix=tok.decode(ids[:PREFIX], clean_up_tokenization_spaces=False)))
    return items

@torch.no_grad()
def score_encoded(tok, model, enc, name):   # enc: (record, index of the first scored token, full token ids)
    tok.padding_side = "right"; pad = tok.pad_token_id
    enc = sorted(enc, key=lambda e: len(e[2])); out = []
    for s in range(0, len(enc), SBS):
        chunk = enc[s:s + SBS]; L = max(len(e[2]) for e in chunk)
        x = torch.full((len(chunk), L), pad); att = torch.zeros((len(chunk), L), dtype=torch.long)
        for i, (_, _, ids) in enumerate(chunk):
            x[i, :len(ids)] = torch.tensor(ids); att[i, :len(ids)] = 1
        logits = model(input_ids=x.cuda(), attention_mask=att.cuda()).logits
        for i, (r, k, ids) in enumerate(chunk):
            n = len(ids)
            lp = torch.log_softmax(logits[i, :n - 1].float(), -1)
            tgt = torch.tensor(ids[1:], device=lp.device)
            p = lp.exp(); e1 = (p * lp).sum(-1)
            arr = torch.stack([-lp.gather(1, tgt[:, None])[:, 0], -e1, (p * lp * lp).sum(-1) - e1 ** 2,
                               p.topk(10, -1).values.sum(-1)]).cpu().numpy().astype(np.float32)[:, k - 1:]
            out.append(dict(id=r["id"], domain=r["domain"], model=name, decoding=r["decoding"], who=r["who"], scorer=name, arr=arr))
    return pd.DataFrame(out)

def score_self(tok, model, recs, name):   # text path: notebook 13's encoding, unchanged
    enc = []
    for r in recs:
        k = len(tok(r["prefix"])["input_ids"])
        ids = tok(r["prefix"] + r["text"], truncation=True, max_length=MAX_TOK)["input_ids"]
        if len(ids) - k >= 10:
            enc.append((r, k, ids))
    return score_encoded(tok, model, enc, name)

def score_ids(tok, model, recs, name):   # id path: BOS (if any) + prefix ids + sampled ids, exactly what generation conditioned on
    B = bos(tok)
    return score_encoded(tok, model, [(r, len(B) + len(r["prefix_ids"]), B + r["prefix_ids"] + r["ids"]) for r in recs], name)

@torch.no_grad()
def generate_batch(model, tok, batch, temp, special):
    tok.padding_side = "left"; pad = tok.pad_token_id; B = bos(tok)
    L = max(len(B) + len(b["prefix_ids"]) for b in batch); new = max(b["target"] for b in batch)
    x = torch.full((len(batch), L), pad); att = torch.zeros((len(batch), L), dtype=torch.long)
    for i, b in enumerate(batch):
        ids = B + b["prefix_ids"]
        x[i, L - len(ids):] = torch.tensor(ids); att[i, L - len(ids):] = 1
    out = model.generate(input_ids=x.cuda(), attention_mask=att.cuda(), do_sample=True, temperature=temp, top_p=1.0, top_k=0,
                         max_new_tokens=new, min_new_tokens=new, suppress_tokens=special, pad_token_id=pad, repetition_penalty=1.0)
    gen = out[:, L:].cpu()
    res = [gen[i, :b["target"]].tolist() for i, b in enumerate(batch)]
    for b, ids in zip(batch, res):   # self-check: exact length, no special tokens
        assert len(ids) == b["target"], ("wrong length", len(ids), b["target"])
        assert not (set(ids) & set(special)), "a special token was generated"
    return res

In [ ]:
# 6. For each model: 300 continuations at T = 1 with the sampled ids kept, then both scoring paths on the same GPU.
for name in MODELS:
    tag = tag_of(name); done = [f"{OUT}/{p}_{tag}.pkl" for p in ("scores_text", "scores_ids")]
    if all(os.path.exists(p) for p in done):
        print("Already done, skipping:", tag); continue
    tok, model = load(name)
    special = sorted(set(tok.all_special_ids) | {tok.eos_token_id} | set(np.atleast_1d(model.generation_config.eos_token_id).tolist() if model.generation_config.eos_token_id is not None else []))
    items = sorted(prepare(tok, sources), key=lambda b: b["target"])
    assert len(items) == 300, f"Only {len(items)} prompts left; some texts are too short"
    torch.manual_seed(SEED); t0 = time.time(); rows = []
    for s in range(0, len(items), BS):
        batch = items[s:s + BS]
        for b, ids in zip(batch, generate_batch(model, tok, batch, TEMP, special)):
            rows.append(dict(id=b["id"], domain=b["domain"], model=name, decoding=f"T{TEMP}", who="ai", prefix=b["prefix"],
                             prefix_ids=b["prefix_ids"], ids=ids, text=tok.decode(ids, clean_up_tokenization_spaces=False), n_tokens=len(ids)))
    json.dump(rows, open(f"{OUT}/gen_{tag}.json", "w"), ensure_ascii=False)
    B = bos(tok); check = []
    for r in rows:   # does notebook 13's re-encoding reproduce the context and the sampled tokens?
        k = len(tok(r["prefix"])["input_ids"]); full = tok(r["prefix"] + r["text"], truncation=True, max_length=MAX_TOK)["input_ids"]
        check.append(dict(id=r["id"], same_prefix=full[:k] == B + r["prefix_ids"], same_continuation=full[k:] == r["ids"], token_count_change=len(full) - k - len(r["ids"])))
    pd.DataFrame(check).to_csv(f"{OUT}/ids_check_{tag}.csv", index=False)
    score_self(tok, model, rows, name).to_pickle(done[0])
    score_ids(tok, model, rows, name).to_pickle(done[1])
    print(f"{tag}: {(time.time() - t0) / 60:.1f} min", flush=True)
    del model; free(name)

In [ ]:
# 7. Quick look (the formal analysis with intervals and the pre-registered decision is scripts/e11_retokenization_diagnostic.py).
rows = []
for name in MODELS:
    tag = tag_of(name)
    c = {p: np.array([(a[1] - a[0]).mean() for a in pd.read_pickle(f"{OUT}/scores_{p}_{tag}.pkl").sort_values("id").arr]) for p in ("text", "ids")}
    chk = pd.read_csv(f"{OUT}/ids_check_{tag}.csv")
    rows.append(dict(model=tag, c_text=c["text"].mean(), c_ids=c["ids"].mean(), diff_text_minus_ids=(c["text"] - c["ids"]).mean(),
                     share_changed=1 - (chk.same_prefix & chk.same_continuation).mean()))
print(pd.DataFrame(rows).round(4).to_string(index=False))

In [ ]:
# 8. Record the run and flush unsynced files back to Google Drive.
run_info = dict(completed_at_utc=datetime.now(timezone.utc).isoformat(), models=MODELS, temperature=TEMP, prefix_tokens=PREFIX, max_tokens=MAX_TOK,
                gen_seed=SEED, humans_manifest_sha256=EXPECTED_MAN, python=sys.version, platform=platform.platform(), torch=torch.__version__,
                transformers=transformers.__version__, gpu=torch.cuda.get_device_name(0))
json.dump(run_info, open(f"{OUT}/run_info.json", "w"), indent=2)
%cd /content
drive.flush_and_unmount()
print("Done: results are in DATA_DIR/14_retokenization_diagnostic/")